# 第七章 文本大数据分析进阶：传统方法与大型语言模型应用
----
书籍名称：《大数据分析：Python 实践与大型语言模型应用》

谢佳松、刘娟



----

## 7.1 情感分析
### 7.1.1 基于词典的情感分析

In [ ]:
try:
    import chardet
except ImportError:
    chardet = None

for dict_path in ["dict/positive_words.txt", "dict/negative_words.txt"]:
    with open(dict_path, "rb") as file:
        data = file.read()
        encoding = chardet.detect(data)["encoding"] if chardet else "utf-8"
    print(dict_path, encoding)


In [ ]:
import jieba
import pandas as pd        
# 加载褒义词典和贬义词典
positive_words = set()
with open('dict/positive_words.txt', 'r', encoding='utf-8') as file:
    for line in file:
        positive_words.add(line.strip())
negative_words = set()
with open('dict/negative_words.txt', 'r', encoding='utf-8') as file:
    for line in file:
        negative_words.add(line.strip())    
# 加载中文停用词表
stopwords = set()
with open( 'dict/中文停用词表.txt', 'r', encoding='utf-8') as file:
    for line in file:
        stopwords.add(line.strip())
# 加载政府工作报告文本数据
with open('data/辽宁2024.txt', 'r', encoding='utf-8') as file:
    report_text = file.read()
# 对文本进行分词，并过滤停用词
words = [word for word in jieba.lcut(report_text) if word not in stopwords]
# 初始化情感得分
sentiment_score = 0
# 遍历分词结果，进行情感极性判断
for word in words:
    # 如果词汇在褒义词典中，增加情感得分
    if word in positive_words:
        sentiment_score += 1
    # 如果词汇在贬义词典中，减少情感得分
    elif word in negative_words:
        sentiment_score -= 1
# 根据情感得分判断文本的整体情感极性
if sentiment_score > 0:
    print("政府工作报告表达了积极情感")
elif sentiment_score < 0:
    print("政府工作报告表达了消极情感")
else:
    print("政府工作报告情感中性")
print (sentiment_score)

### 7.1.2 基于机器学习的情感分析

In [ ]:
# 导入所需库
from sklearn.feature_extraction.text import CountVectorizer  # 文本特征提取
from sklearn.naive_bayes import MultinomialNB                # 朴素贝叶斯分类器
from sklearn.pipeline import make_pipeline                  # 构建机器学习管道
from sklearn.metrics import accuracy_score                  # 计算模型准确率
# 1. 准备带有情感标签的训练数据
data = [
    ("这部电影太棒了！", "positive"),
    ("这个产品很失望。", "negative"),
    ("服务质量很好。", "positive"),
    ("我不喜欢这个设计。", "negative"),
    ("这个城市很美丽。", "positive"),
    ("这个餐厅的食物很难吃。", "negative")
]
# 拆分特征 (X) 和标签 (y)
X, y = zip(*data)
# 2. 构建机器学习管道
# CountVectorizer 将文本转换为词频向量
# MultinomialNB 是基于词频的概率分类器
model = make_pipeline(CountVectorizer(), MultinomialNB())
# 3. 在训练集上训练模型
model.fit(X, y)
# 4. 在训练集上预测
y_pred = model.predict(X)
# 5. 评估模型准确率
accuracy = accuracy_score(y, y_pred)
print("分类器准确率:", accuracy)
# 6. 对新文本进行预测
new_texts = ["这本书很不好看。", "我对这个产品感到很失望。"]
predicted_sentiments = model.predict(new_texts)
print("新文本的情感极性:", predicted_sentiments)

In [ ]:
# 第一步：导入所需库
from sklearn.feature_extraction.text import TfidfVectorizer   # 用于 TF-IDF 特征提取
from sklearn.svm import LinearSVC                            # 线性支持向量机
from sklearn.pipeline import make_pipeline                   # 构建管道
from sklearn.metrics import classification_report            # 性能评估报告

# 第二步：准备带标签的训练数据
data = [
    ("这部电影太棒了！", "positive"),
    ("这个产品很失望。", "negative"),
    ("服务质量很好。", "positive"),
    ("我不喜欢这个设计。", "negative"),
    ("这个城市很美丽。", "positive"),
    ("这个餐厅的食物很难吃。", "negative"),
    ("这次体验非常糟糕。", "negative"),
    ("这款手机的拍照效果很好。", "positive")
]

# 将文本和标签分离
X, y = zip(*data)

# 第三步：构建管道（TF-IDF + SVM）
model = make_pipeline(TfidfVectorizer(), LinearSVC())

# 第四步：训练模型
model.fit(X, y)

# 第五步：评估模型
y_pred = model.predict(X)
print("分类报告：\n", classification_report(y, y_pred))

# 第六步：预测新文本情感
new_texts = ["这部电影剧情很精彩！", "这个手机太差了，电池续航很糟糕。"]
predictions = model.predict(new_texts)
print("新文本情感预测：", list(zip(new_texts, predictions)))

### 7.1.3 基于大型语言模型的情感分析

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline
# Step 1: 加载预训练的中文 BERT 模型
# 'uer/roberta-base-finetuned-jd-binary-chinese' 是在京东评论上微调的中文情感模型
model_name = "uer/roberta-base-finetuned-jd-binary-chinese"
classifier = pipeline("sentiment-analysis", model=model_name, tokenizer=model_name)
# Step 2: 定义待分析的文本
texts = [
    "这款手机的续航能力非常出色，我很满意！",
    "客服态度差，物流也很慢，体验非常不好。",
    "这家公司虽然今年盈利下降，但整体表现超出预期。"
]
# Step 3: 执行情感预测
results = classifier(texts)
# Step 4: 输出结果
for text, res in zip(texts, results):
    print(f"文本: {text}")
    print(f"预测情感: {res['label']}, 置信度: {res['score']:.4f}")

In [ ]:
import os
API_BASE = "https://api.lingyiwanwu.com/v1"
API_KEY = os.getenv("Yi_KEY", "")
if not API_KEY or API_KEY == "your_keys":
    print("Yi API key 未配置，跳过在线情感分析示例。")
else:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY, base_url=API_BASE)
        texts = ["这款智能音箱的音质非常出色，我很喜欢！", "客服态度差，问题迟迟得不到解决。", "这个产品一般般，没有想象中好。"]
        for text in texts:
            prompt = f"请对以下文本进行情感分析，判断其情绪是正面还是负面，并用一句话简要说明理由。文本：{text}"
            response = client.chat.completions.create(model="yi-lightning", messages=[{"role": "user", "content": prompt}], max_tokens=200)
            print(f"文本：{text}")
            print("情感分析结果：", response.choices[0].message.content)
            print("-" * 40)
    except Exception as e:
        print("Yi 在线情感分析示例无法运行:", e)


## 7.2 文本相似度
### 7.2.1 基于词典的文本相似度分析

In [ ]:
import jieba
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# 示例两段文本
text1 = "经济复苏带来就业机会"
text2 = "经济回暖增加就业岗位"
# 使用jieba分词
text1_cut = " ".join(jieba.lcut(text1))
text2_cut = " ".join(jieba.lcut(text2))
# 转换为词频向量
vectorizer = CountVectorizer()
vectors = vectorizer.fit_transform([text1_cut, text2_cut])
# 计算余弦相似度
similarity = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
print(f"相似度: {similarity:.4f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import jieba

# 中文分词
text1_cut = " ".join(jieba.lcut("经济复苏带来就业机会"))
text2_cut = " ".join(jieba.lcut("经济回暖增加就业岗位"))

# TF-IDF向量化
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([text1_cut, text2_cut])

# 计算相似度
similarity = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
print(f"TF-IDF相似度: {similarity:.4f}")

###  7.2.2 基于机器学习的文本相似度分析

In [ ]:
import jieba
import numpy as np
from gensim.models import KeyedVectors, Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import os
def load_w2v_model(model_path):
    """
    自动加载 Word2Vec 模型，支持 .bin, .txt/.vec, .model 格式
    """
    _, ext = os.path.splitext(model_path)
    
    if ext == ".bin":  # 二进制格式
        print("Loading binary Word2Vec model...")
        return KeyedVectors.load_word2vec_format(model_path, binary=True)
    
    elif ext in [".txt", ".vec"]:  # 文本格式
        print("Loading text Word2Vec model...")
        return KeyedVectors.load_word2vec_format(model_path, binary=False)
    
    elif ext == ".model":  # Gensim 保存的模型
        print("Loading Gensim .model Word2Vec...")
        return Word2Vec.load(model_path).wv
    
    else:
        raise ValueError(f"Unsupported model format: {ext}")
def get_sentence_vector(words, model):
    """计算句子中词向量的平均值"""
    if len(words) == 0:
        return np.zeros(model.vector_size)
    return np.mean([model[w] for w in words if w in model], axis=0)

# ============ 主程序 ============
# 1. 加载模型（自动识别类型）
model_path = "model/wiki.zh.text.model"
w2v_model = load_w2v_model(model_path)
# 2. 文本输入
text1 = "经济复苏带来就业机会"
text2 = "经济回暖增加就业岗位"
# 3. 中文分词
words1 = [w for w in jieba.lcut(text1) if w in w2v_model]
words2 = [w for w in jieba.lcut(text2) if w in w2v_model]
# 4. 获取句子向量
vec1 = get_sentence_vector(words1, w2v_model).reshape(1, -1)
vec2 = get_sentence_vector(words2, w2v_model).reshape(1, -1)
# 5. 计算余弦相似度
similarity = cosine_similarity(vec1, vec2)[0][0]
print(f"Word2Vec相似度：{similarity:.4f}")

### 7.2.3 基于大型语言模型的文本相似度分析

In [ ]:
from sentence_transformers import SentenceTransformer, util
#设定模型：all-MiniLM-L6-v2
model = SentenceTransformer("all-MiniLM-L6-v2") #导入 Sentence Transformers 库中的 SentenceTransformer 类，它用于加载预训练的句子嵌入模型

# 词嵌入
emb1 = model.encode("经济复苏带来就业机会")
emb2 = model.encode("经济回暖增加就业岗位")
# 计算相似度
cos_sim = util.cos_sim(emb1, emb2)
print("Cosine-Similarity:", cos_sim)

In [ ]:
from sentence_transformers import SentenceTransformer, util# 导入了 Sentence Transformers 库中的 SentenceTransformer 类，它用于加载预训练的句子嵌入模型
model = SentenceTransformer("all-MiniLM-L6-v2")#使用了名为 "all-MiniLM-L6-v2" 的预训练模型。这个模型是基于 MiniLM 架构，是一个轻量级的语言模型。它被训练用于生成句子的语义嵌入向量。
#词嵌入
#将句子列表传递给模型的 encode() 方法，以获取它们的嵌入向量。模型将每个句子转换为其语义空间中的向量表示
emb1 = model.encode("东北财经大学") 
emb2 = model.encode("上海财经大学")
emb3 = model.encode("中山大学")
#根据词嵌入计算句子相似性
cos_sim = util.cos_sim(emb1, emb2)
cos_sim1 = util.cos_sim(emb1, emb3)
print("Cosine-Similarity:", cos_sim)
print("Cosine-Similarity:", cos_sim1)

In [ ]:
# 观察SentenceTransformer模型的基本信息
model

In [ ]:
import numpy as np
norms = np.linalg.norm(emb1)  #观察到嵌入向量的模是否接近于 1，从而判断它们是否已经被正则化。
print("Embeddings norms:", norms)

In [ ]:
#导入SentenceTransformer和models模块。SentenceTransformer是一个高级API，用于创建句子嵌入模型。models模块包含了各种预训练的模型和层。
from sentence_transformers import SentenceTransformer, models
#创建了一个基于BERT的Transformer模型。其中：
#bert-base-uncased参数指定了使用预训练的BERT基础模型
#max_seq_length参数设置了输入句子的最大长度，BERT的原始模型支持的最大长度是512，这里出于性能考虑设置为256
word_embedding_model = models.Transformer("bert-base-uncased", max_seq_length=256)
#创建了一个Pooling模型,用于从Transformer模型中获取句子表示。
#get_word_embedding_dimension()方法用于获取Transformer模型输出的单词嵌入维度，这个维度将作为Pooling模型的输入维度
#注意：仅仅获取词嵌入模型的词嵌入维度，并未定义池化的方式，因此是默认的平均池化法
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
#最后，创建了一个SentenceTransformer模型，它包含了两个参数：word_embedding_model和pooling_model。
#未加标准化层
#这个模型可以将句子转换为固定长度的向量表示，此时包含两层，并未包含基准模型中的第三层，正则层
model_try = SentenceTransformer(modules=[word_embedding_model, pooling_model])


In [ ]:
model_try

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
sentences = ["大数据分析：Python实践与大型语言模型应用"]
# 获取嵌入向量
embeddings = model_try.encode(sentences)
#查看向量维度
print("Embeddings shape:", embeddings. shape)
# 计算嵌入向量的模
norms = np.linalg.norm(embeddings, axis=1)  
print("Embeddings norms:", norms)

In [ ]:
from sentence_transformers import SentenceTransformer, models
from torch import nn
# embedding 层
word_embedding_model = models.Transformer("bert-base-uncased", max_seq_length=256)
# pooling 层
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
#增加密集连接层
dense_model = models.Dense(
    in_features=pooling_model.get_sentence_embedding_dimension(), #设置为池化模型输出的句子向量维度
    out_features=256,   #设定输出为256维度
    activation_function=nn.Tanh(), #引入非线性变化，激活密集连接层
)
#设定模型
model1= SentenceTransformer(modules=[word_embedding_model, pooling_model, dense_model])

In [ ]:
from sentence_transformers import SentenceTransformer, util
emb1 = model1.encode("东北财经大学")
emb2 = model1.encode("上海财经大学")
emb3 = model_try.encode("东北财经大学")
emb4 = model_try.encode("上海财经大学")
emb5 = model.encode("东北财经大学")
emb6 = model.encode("上海财经大学")
cos_sim1 = util.cos_sim(emb1, emb2)
cos_sim2 = util.cos_sim(emb3, emb4)
cos_sim3= util.cos_sim(emb5, emb6)
print("Cosine-Similarity:", cos_sim1,cos_sim2,cos_sim3)

## 7.3 主题建模
### 7.3.1 传统主题建模方法：LDA

In [ ]:
import jieba
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd

# 样本文本
docs = [
    "公司今年加大了技术创新的投入",
    "市场竞争激烈，企业需要加强管理",
    "政府鼓励绿色环保项目的发展",
    "金融风险是企业面临的主要挑战",
    "技术研发是企业成长的动力"
]
# 1. 分词
tokenized_docs = [" ".join(jieba.lcut(doc)) for doc in docs]

# 2. 文本转词袋矩阵
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(tokenized_docs)

# 3. 训练 LDA 模型
lda_model = LatentDirichletAllocation(n_components=3, random_state=0)
lda_model.fit(X)

# 4. 输出每个主题的关键词
words = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_model.components_):
    print(f"主题 {topic_idx}: ", [words[i] for i in topic.argsort()[:-6:-1]])

In [ ]:
# 每个文档的主题分布
doc_topic_dist = lda_model.transform(X)
doc_topic_df = pd.DataFrame(doc_topic_dist, columns=[f'主题_{i}' for i in range(3)])
doc_topic_df['文档'] = docs
# 查看每个文档的主要主题
doc_topic_df['主要主题'] = doc_topic_df.iloc[:, :-1].idxmax(axis=1)
print(doc_topic_df)

### 7.3.2 基于大型语言模型的主题建模方法：BERTopic

In [ ]:
#下载数据
from sklearn.datasets import fetch_20newsgroups
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']

In [ ]:
for i in range(5):
    print("\nDocument", i+1, ":", docs[i])

In [ ]:
from bertopic import BERTopic
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(docs)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(0)


In [ ]:
topic_model.get_document_info(docs)

In [ ]:
from bertopic.representation import KeyBERTInspired
#调整主题表示
representation_model = KeyBERTInspired()
topic_model = BERTopic(representation_model=representation_model)

In [ ]:
import openai
from bertopic.representation import OpenAI

client = openai.OpenAI(api_key="sk-...")
representation_model = OpenAI(client, model="gpt-4o-mini", chat=True)
topic_model = BERTopic(representation_model=representation_model)

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_distribution(probs[200], min_probability=0.015)

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [ ]:
topic_model.visualize_heatmap(n_clusters=20, width=1000, height=1000)

## 7.4 文本复杂度
### 7.4.1 传统与机器学习文本复杂度测量方法

In [ ]:
import jieba
import re
def calculate_chinese_complexity(text):
    # 分句
    sentences = re.split('[。！？]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    if not sentences:
        return 0, 0
    # 平均句子长度
    char_counts = [len(s) for s in sentences]
    avg_char_length = sum(char_counts) / len(sentences)
    # 复杂词比例（长度>2的词视为复杂词）
    word_list = jieba.lcut(text)
    complex_words = sum(1 for w in word_list if len(w) > 2)
    percent_complex = (complex_words / len(word_list)) * 100 if word_list else 0
    # 调整后的 Fog Index
    fog_index = 0.4 * (avg_char_length + percent_complex)
    return avg_char_length, fog_index
text = "随着科技的发展与互联网的完善，品牌发展将线下与线上融为一体已经是一种趋势。对此，公司适时外聘咨询机构，构筑了全渠道布局，将线下实体与线上PC端、移动端打通，建立微商场，从而实现了三方渠道资源的融合与共享。"
length, fog = calculate_chinese_complexity(text)
print(f"平均字符长度: {length:.2f}")
print(f"调整后 Fog Index: {fog:.2f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
import numpy as np
import jieba

# 模拟训练数据（经济管理领域文本及复杂度评分）
texts = [
    "公司经营情况良好，利润稳步增长。",
    "该财务报告包含大量会计准则解释及附注说明，阅读难度较高。",
    "市场分析显示消费趋势变化，建议关注风险控制。"
]
complexity_scores = [1, 3, 2]  # 人为标注复杂度：1简单，3复杂

# 中文分词处理
def chinese_tokenizer(text):
    return jieba.lcut(text)

# 特征提取：使用 TF-IDF 向量化文本
vectorizer = TfidfVectorizer(tokenizer=chinese_tokenizer)
X = vectorizer.fit_transform(texts)
y = np.array(complexity_scores)

# 训练线性回归模型
model = LinearRegression()
model.fit(X, y)

# 预测新文本复杂度
new_text = "本年度财务披露涉及复杂的风险管理策略与税收筹划细节。"
new_X = vectorizer.transform([new_text])
predicted_complexity = model.predict(new_X)[0]
print(f"预测复杂度: {predicted_complexity:.2f}")

In [ ]:
import jieba
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
import matplotlib.font_manager as fm

texts = [
    "公司经营情况良好，利润稳步增长。",
    "该财务报告包含大量会计准则解释及附注说明，阅读难度较高。",
    "市场分析显示消费趋势变化，建议关注风险控制。",
    "本公告包含复杂的金融衍生品说明，涉及对冲策略与风险管理措施。",
    "董事会提出了创新的数字化转型战略，技术架构升级方案较为详细。"
]
# 人为设定的复杂度标签（1=简单，5=复杂）
complexity_scores = [1, 4, 2, 5, 3]
#分词
def chinese_tokenizer(text):
    return jieba.lcut(text)

# 随机森林训练
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)
#
# 批量预测新文本
new_texts = [
    "年度报告包含详细财务指标，条理清晰。",
    "涉及海外投资与复杂的跨国税务筹划，披露较为晦涩。",
    "公司未来战略围绕数字化和人工智能展开，说明简洁明了。"
]
new_X = vectorizer.transform(new_texts)
predicted_complexities = model.predict(new_X)

# 6. 可视化预测结果）

font_path = 'font/Songti.ttc'
my_font = fm.FontProperties(fname=font_path)

plt.figure(figsize=(10, 6))
plt.bar(range(len(new_texts)), predicted_complexities, color='skyblue')
plt.xticks(range(len(new_texts)), new_texts, fontproperties=my_font, rotation=20)
plt.ylabel("预测复杂度评分", fontproperties=my_font, fontsize=12)
plt.title("中文文本复杂度预测（随机森林模型）", fontproperties=my_font, fontsize=14)
plt.tight_layout()
plt.show()

### 7.4.2基于大型语言模型测量文本复杂度

In [ ]:
from transformers import AutoTokenizer, AutoModel  # 用于加载 RoBERTa 分词器和模型
import torch
from sklearn.linear_model import LinearRegression  # 回归模型
import numpy as np

# 加载中文 RoBERTa 模型
model_name = "hfl/chinese-roberta-wwm-ext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # 设为评估模式，避免更新参数

# 定义函数：获取文本的向量表示
def get_embedding(text):
    """
    使用RoBERTa提取文本的句向量（CLS向量）。
    :param text: 输入文本
    :return: 768维的句向量
    """
    # 编码文本 -> 模型输入张量
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    # 取 [CLS] 位置的向量表示
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    return embedding
# 构建训练数据
texts = [
    "经济增长带来就业机会。",
    "复杂的财务披露包含大量技术术语和冗长句子。",
    "公司治理结构清晰，年报信息透明。",
    "金融市场波动加剧，披露内容晦涩难懂。"
]
# 人工标注的复杂度分数（1-5分）
complexity_scores = [2, 4, 1, 5]

# 将文本转换为向量
X = np.array([get_embedding(text) for text in texts])
y = np.array(complexity_scores)

#  训练线性回归模型
regressor = LinearRegression()
regressor.fit(X, y)

#预测
new_text = "公司年度报告因技术细节复杂难以理解。"
new_X = np.array([get_embedding(new_text)])
predicted_complexity = regressor.predict(new_X)[0]
print(f"预测复杂度: {predicted_complexity:.2f}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

# 加载微调模型
model_name = "hfl/chinese-roberta-wwm-ext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
model.train()  # 初始为训练模式

# 自定义数据集
class ComplexityDataset(Dataset):
    def __init__(self, texts, scores, tokenizer):
        self.encodings = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        self.scores = torch.tensor(scores, dtype=torch.float32).unsqueeze(1)  # 确保维度匹配

    def __len__(self): return len(self.scores)
    def __getitem__(self, idx): return {key: val[idx] for key, val in self.encodings.items()}, self.scores[idx]

# 扩展训练数据
texts = [
    "经济增长带来就业机会。",
    "复杂的财务披露包含大量技术术语和冗长句子。",
    "公司治理结构清晰，年报信息透明。",
    "金融市场波动加剧，披露内容晦涩难懂。",
    "政策支持中小企业发展，效果显著。",
    "年度财务报告因法规要求复杂化。",
    "市场复苏促进就业率提升。",
    "技术创新推动经济转型，报告详尽。"
]
complexity_scores = [2, 4, 1, 5, 2, 3, 2, 4]  # 扩展标签

# 划分训练集和验证集
train_texts, val_texts, train_scores, val_scores = train_test_split(texts, complexity_scores, test_size=0.2, random_state=42)
train_dataset = ComplexityDataset(train_texts, train_scores, tokenizer)
val_dataset = ComplexityDataset(val_texts, val_scores, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2)

# 微调模型
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-2)
for epoch in range(5):  # 增加 epoch
    model.train()
    train_loss = 0
    for batch, (inputs, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print(f"Epoch {epoch+1}, Train Loss: {train_loss/len(train_loader):.4f}")

    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(**inputs, labels=labels)
            val_loss += outputs.loss.item()
    print(f"Epoch {epoch+1}, Val Loss: {val_loss/len(val_loader):.4f}")

# 预测函数
def predict_complexity(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits.squeeze().item()

# 预测新文本
new_text = "公司年度报告因技术细节复杂难以理解。"
predicted_complexity = predict_complexity(new_text)
print(f"预测复杂度: {predicted_complexity:.2f}")

# 修正困惑度计算（使用语言模型损失）
from transformers import AutoModelForMaskedLM
lm_model = AutoModelForMaskedLM.from_pretrained(model_name)
def calculate_perplexity(text):
    lm_model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = lm_model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss
        if torch.isnan(loss) or torch.isinf(loss):
            return float('inf')
        return torch.exp(loss).item()

perplexity = calculate_perplexity(new_text)
print(f"困惑度 (复杂度指标): {perplexity:.2f}")

In [ ]:
import os
from openai import OpenAI
# ================================
# 1. 初始化零一万物 API 客户端
# ================================
API_BASE = "https://api.lingyiwanwu.com/v1"
API_KEY = "your_keys"  # 请替换为你的真实API Key

client = OpenAI(
    api_key=API_KEY,
    base_url=API_BASE
)
# ================================
# 2. 定义待评估的中文文本
# ================================
text = """
随着宏观经济环境的变化，公司通过并购重组拓展海外市场，
积极运用财务杠杆提高资本回报率，但同时面临汇率波动与流动性风险的挑战。
"""
# ================================
# 3. 构建提示词（Prompt）
# ================================
prompt = f"""
请分析以下文本的复杂度，并给出 1-5 的评分（1=非常易读，5=非常难读）。
请简要说明评分理由。
文本：{text}
"""
# ================================
# 4. 调用零一万物模型
# ================================
response = client.chat.completions.create(
    model="yi-lightning",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200
)
# ================================
# 5. 输出结果
# ================================
print("模型返回：")
print(response.choices[0].message.content)

## 7.5 从传统方法到大语言模型应用：性能对比与混合策略


* 混合策略

In [ ]:
"""
情感分析对比：词典法、TF-IDF+NB、BERT、混合策略、Yi LLM、Yi混合策略
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import torch
from transformers import AutoTokenizer, AutoModel
from openai import OpenAI

# -------------------------------
# 1. 数据集（包含隐含情感与反讽）
# -------------------------------
data = [
    ("这家餐厅真是太棒了！", "positive"),
    ("这个产品令人失望。", "negative"),
    ("我对这部电影印象很好。", "positive"),
    ("天气很糟糕，我一点都不开心。", "negative"),
    ("服务态度非常周到，值得表扬。", "positive"),
    ("真是好极了，等了两个小时才上菜。", "negative"),  # 反讽
    ("老板笑容可掬，食物却难以下咽。", "negative"),
    ("电影剧情平淡无奇，没有什么亮点。", "negative"),
    ("这款手机的外观设计简直太漂亮了！", "positive"),
    ("勉强还行，没有想象中的那么差。", "positive"),  # 隐含正向
    ("好家伙，这车油耗真感人啊。", "negative"),  # 反讽
    ("价格公道，性价比很高，非常推荐。", "positive"),
    ("虽然手机配置一般，但售后服务很到位。", "positive"),  # 复杂表达
    ("这部电影本来很期待，但剧情拖沓。", "negative"),
    ("客服态度差得让我怀疑人生。", "negative"),
    ("设计得中规中矩，没有亮点。", "negative"),
    ("报告内容详尽，逻辑清晰。", "positive"),
    ("虽然不完美，但整体体验还不错。", "positive"),  # 隐含正向
    ("商家的解释很耐心，虽然问题没解决。", "positive"),
    ("真是让我无语，差评！", "negative")
]
df = pd.DataFrame(data, columns=["text", "label"])
X = df["text"].values
y = df["label"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# -------------------------------
# 2. 方法1：词典法
# -------------------------------
pos_dict = ["棒", "很好", "周到", "漂亮"]
neg_dict = ["失望", "糟糕", "差" "无奇"]

def lexicon_predict(texts):
    preds = []
    for t in texts:
        score = 0
        for p in pos_dict:
            if re.search(p, t): score += 1
        for n in neg_dict:
            if re.search(n, t): score -= 1
        preds.append("positive" if score > 0 else "negative")
    return preds

y_pred_dict = lexicon_predict(X_test)
acc_dict = accuracy_score(y_test, y_pred_dict)
print(f"方法1 – 词典法准确率: {acc_dict:.2f}")

# -------------------------------
# 3. 方法2：TF-IDF + 朴素贝叶斯
# -------------------------------
tfidf = TfidfVectorizer(max_features=100)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f"方法2 – TF-IDF+NB准确率: {acc_nb:.2f}")

# -------------------------------
# 4. 方法3：BERT + 逻辑回归
# -------------------------------
X_train_bert = get_bert_embeddings(X_train)
X_test_bert = get_bert_embeddings(X_test)

# 动态选择 PCA 维度
pca_dim = min(8, X_train_bert.shape[1])  # 不超过样本数
pca = PCA(n_components=pca_dim, random_state=42)
X_train_bert = pca.fit_transform(X_train_bert)
X_test_bert = pca.transform(X_test_bert)

bert_lr = LogisticRegression(max_iter=1000)
bert_lr.fit(X_train_bert, y_train)
y_pred_bert = bert_lr.predict(X_test_bert)
acc_bert = accuracy_score(y_test, y_pred_bert)
print(f"方法3 – BERT准确率: {acc_bert:.2f}")

# -------------------------------
# 5. 方法4：混合策略（TF-IDF + BERT + 随机森林）
# -------------------------------
X_train_mix = np.hstack([X_train_tfidf.toarray(), X_train_bert])
X_test_mix = np.hstack([X_test_tfidf.toarray(), X_test_bert])

scaler = MinMaxScaler()
X_train_mix = scaler.fit_transform(X_train_mix)
X_test_mix = scaler.transform(X_test_mix)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_mix, y_train)
y_pred_mix = rf.predict(X_test_mix)
acc_mix = accuracy_score(y_test, y_pred_mix)
print(f"方法4 – 混合策略准确率: {acc_mix:.2f}")

# -------------------------------
# 6. 方法5：Yi LLM API
# -------------------------------
API_BASE = "https://api.lingyiwanwu.com/v1"
API_KEY = "your_keys"  # 请替换
client = OpenAI(api_key=API_KEY, base_url=API_BASE)

def yi_sentiment(text):
    prompt = f"请判断以下文本的情感是正面还是负面：{text}，只输出'正面'或'负面'"
    try:
        response = client.chat.completions.create(
            model="yi-lightning",
            messages=[{"role": "user", "content": prompt}]
        )
        ans = response.choices[0].message.content
        return "positive" if "正面" in ans else "negative"
    except Exception as e:
        print(f"Yi LLM调用失败: {e}")
        return "positive"

y_pred_yi = [yi_sentiment(t) for t in X_test]
acc_yi = accuracy_score(y_test, y_pred_yi)
print(f"方法5 – Yi LLM准确率: {acc_yi:.2f}")

# -------------------------------
# 7. 方法6：Yi LLM混合策略
# -------------------------------
y_train_yi = [yi_sentiment(t) for t in X_train]
y_test_yi = [yi_sentiment(t) for t in X_test]
yi_train_features = np.array([1 if p == "positive" else 0 for p in y_train_yi]).reshape(-1, 1)
yi_test_features = np.array([1 if p == "positive" else 0 for p in y_test_yi]).reshape(-1, 1)

yi_train_features = np.repeat(yi_train_features, 10, axis=1)
yi_test_features = np.repeat(yi_test_features, 10, axis=1)

X_train_mix = np.hstack([X_train_tfidf.toarray(), X_train_bert, yi_train_features])
X_test_mix = np.hstack([X_test_tfidf.toarray(), X_test_bert, yi_test_features])

scaler = MinMaxScaler()
X_train_mix = scaler.fit_transform(X_train_mix)
X_test_mix = scaler.transform(X_test_mix)

rf_yi = RandomForestClassifier(n_estimators=200, random_state=42)
rf_yi.fit(X_train_mix, y_train)

y_pred_yi_mix = rf_yi.predict(X_test_mix)
acc_yi_mix = accuracy_score(y_test, y_pred_yi_mix)
print(f"方法6 – Yi混合策略准确率: {acc_yi_mix:.2f}")

# -------------------------------
# 8. 可视化结果
# -------------------------------
accuracies = [acc_dict, acc_nb, acc_bert, acc_mix, acc_yi, acc_yi_mix]
methods = ["词典法", "TF-IDF+NB", "BERT", "混合策略", "Yi LLM", "Yi混合策略"]

plt.figure(figsize=(8, 5))
plt.bar(methods, accuracies, color=['#66c2a5','#fc8d62','#8da0cb','#e78ac3','#a6d854','#ffd92f'])
plt.ylim(0, 1)
plt.ylabel("准确率")
plt.title("情感分析方法对比")
plt.xticks(rotation=30)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

# 加载字体文件
font_path = 'font/Songti.ttc'
my_font = fm.FontProperties(fname=font_path)

# 设置全局字体（可选，确保所有文本使用中文字体）
plt.rcParams['font.family'] = my_font.get_name()
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 数据
methods = ['传统方法', 'LLM方法', '混合策略']
accuracy = [70, 88, 82]
time_minutes = [10, 40, 20]
resource_usage = [30, 100, 60]  # 假设百分比

# 创建图形
plt.figure(figsize=(10, 5))

# 子图 1：准确率对比
plt.subplot(1, 2, 1)
plt.bar(methods, accuracy, color=['blue', 'orange', 'green'])
plt.title('准确率比较', fontproperties=my_font)
plt.ylabel('准确率 (%)', fontproperties=my_font)
# 手动设置 x 轴标签字体
plt.xticks(methods, fontproperties=my_font)

# 子图 2：时间对比
plt.subplot(1, 2, 2)
plt.bar(methods, time_minutes, color=['blue', 'orange', 'green'])
plt.title('时间比较', fontproperties=my_font)
plt.ylabel('时间 (分钟)', fontproperties=my_font)
# 手动设置 x 轴标签字体
plt.xticks(methods, fontproperties=my_font)

# 调整布局并保存
plt.tight_layout()
plt.show()